In [1]:
import pandas as pd
df = pd.read_csv('participants.tsv', sep='\t')

# 1. Diagnosis counts
print(df['diagnosis'].value_counts(), '\n')

# 2. Prefix rule check
print(pd.crosstab(df['participant_id'].str[4:6], df['diagnosis']), '\n')

# 3. What the modality flags actually contain
print("dwi column raw values:", df['dwi'].unique())
print("T1w column raw values:", df['T1w'].unique(), '\n')

# 4. The number that matters: who has both DWI and T1
has_dwi = df['dwi'].notna()
has_t1  = df['T1w'].notna()
print(pd.crosstab(df['diagnosis'], has_dwi & has_t1), '\n')

# 5. Artifact flag distribution
print(pd.crosstab(df['diagnosis'], df['ghost_NoGhost']))

diagnosis
CONTROL    130
SCHZ        50
BIPOLAR     49
ADHD        43
Name: count, dtype: int64 

diagnosis       ADHD  BIPOLAR  CONTROL  SCHZ
participant_id                              
10                 0        0      100     0
11                 0        0       30     0
50                 0        0        0    50
60                 0       49        0     0
70                43        0        0     0 

dwi column raw values: [ 1. nan]
T1w column raw values: [ 1. nan] 

col_0      False  True 
diagnosis              
ADHD           2     41
BIPOLAR        0     49
CONTROL        7    123
SCHZ           1     49 

ghost_NoGhost  No_ghost  ghost
diagnosis                     
ADHD                 36      7
BIPOLAR              41      8
CONTROL             110     20
SCHZ                 30     20


In [2]:
import pandas as pd
df = pd.read_csv('participants.tsv', sep='\t')

sub = df[df['diagnosis'].isin(['CONTROL','SCHZ'])].copy()

# Which requirement costs what?
print(pd.crosstab(sub['diagnosis'], sub['dwi'].notna(),  rownames=['dx'], colnames=['has_dwi']), '\n')
print(pd.crosstab(sub['diagnosis'], sub['T1w'].notna(), rownames=['dx'], colnames=['has_T1']), '\n')

# Cohort on DWI alone
coh = sub[sub['dwi'].notna()].copy()
print("Eligible on DWI only:")
print(coh['diagnosis'].value_counts(), '\n')

# Can we even match? Age and sex distributions
print(coh.groupby('diagnosis')['age'].describe()[['count','mean','std','min','max']], '\n')
print(pd.crosstab(coh['diagnosis'], coh['gender']))

has_dwi  False  True 
dx                   
CONTROL      7    123
SCHZ         1     49 

has_T1   False  True 
dx                   
CONTROL      5    125
SCHZ         0     50 

Eligible on DWI only:
diagnosis
CONTROL    123
SCHZ        49
Name: count, dtype: int64 

           count       mean       std   min   max
diagnosis                                        
CONTROL    123.0  31.601626  8.845430  21.0  50.0
SCHZ        49.0  36.244898  8.837728  22.0  49.0 

gender      F   M
diagnosis        
CONTROL    58  65
SCHZ       12  37


In [3]:
import pandas as pd, numpy as np
from scipy import stats

df = pd.read_csv('participants.tsv', sep='\t')
coh = df[df['diagnosis'].isin(['CONTROL','SCHZ']) & df['dwi'].notna()].copy()

scz  = coh[coh['diagnosis']=='SCHZ'].copy()
ctrl = coh[coh['diagnosis']=='CONTROL'].copy()

# Match hardest cases first: extreme ages have fewest candidates
scz = scz.reindex(scz['age'].sub(scz['age'].mean()).abs().sort_values(ascending=False).index)

used, pairs = set(), []
for _, p in scz.iterrows():
    cand = ctrl[(ctrl['gender']==p['gender']) & (~ctrl['participant_id'].isin(used))]
    if cand.empty:
        print("NO MATCH for", p['participant_id'], p['gender'], p['age']); continue
    m = cand.iloc[(cand['age']-p['age']).abs().argmin()]
    used.add(m['participant_id'])
    pairs.append({'scz_id':p['participant_id'],'scz_age':p['age'],
                  'ctrl_id':m['participant_id'],'ctrl_age':m['age'],
                  'gender':p['gender'],'diff':abs(p['age']-m['age'])})

pairs = pd.DataFrame(pairs)
print(f"\nmatched {len(pairs)} pairs")
print("age diff within pair: mean %.2f  max %d" % (pairs['diff'].mean(), pairs['diff'].max()))
print((pairs['diff']>5).sum(), "pairs differ by >5 years\n")

sel = coh[coh['participant_id'].isin(pairs['scz_id']) | coh['participant_id'].isin(pairs['ctrl_id'])].copy()
print(sel.groupby('diagnosis')['age'].agg(['count','mean','std']), '\n')
print(pd.crosstab(sel['diagnosis'], sel['gender']), '\n')

a = sel[sel['diagnosis']=='SCHZ']['age']; b = sel[sel['diagnosis']=='CONTROL']['age']
print("age t-test  p = %.3f" % stats.ttest_ind(a,b).pvalue)
print("sex chi2    p = %.3f" % stats.chi2_contingency(pd.crosstab(sel['diagnosis'], sel['gender']))[1])
print(pd.crosstab(sel['diagnosis'], sel['ghost_NoGhost']))

sel.to_csv('cohort.tsv', sep='\t', index=False)
pairs.to_csv('matched_pairs.tsv', sep='\t', index=False)


matched 49 pairs
age diff within pair: mean 0.65  max 3
0 pairs differ by >5 years

           count       mean       std
diagnosis                            
CONTROL       49  35.755102  8.511097
SCHZ          49  36.244898  8.837728 

gender      F   M
diagnosis        
CONTROL    12  37
SCHZ       12  37 

age t-test  p = 0.781
sex chi2    p = 1.000
ghost_NoGhost  No_ghost  ghost
diagnosis                     
CONTROL              43      6
SCHZ                 29     20


In [4]:
import json, pandas as pd
rows = []
for s in pd.read_csv('cohort.tsv', sep='\t')['participant_id']:
    d = json.load(open(f'raw/{s}/dwi/{s}_dwi.json'))
    rows.append({'sub': s, 'pe': d.get('PhaseEncodingDirection'),
                 'trt': round(d.get('TotalReadoutTime', float('nan')), 6),
                 'te': d.get('EchoTime'), 'tr': d.get('RepetitionTime'),
                 'nslices': len(d.get('SliceTiming', [])),
                 'serial': d.get('DeviceSerialNumber')})
p = pd.DataFrame(rows)
for c in ['pe','trt','te','tr','nslices','serial']:
    print(c, p[c].value_counts().to_dict())
p.to_csv('acq_params.tsv', sep='\t', index=False)

pe {'j-': 98}
trt {0.088318: 98}
te {0.0926: 66, 0.093: 32}
tr {8.4: 53, 9.0: 32, 7.0: 13}
nslices {60: 85, 50: 13}
serial {'35343': 66, '35426': 32}


In [6]:
import pandas as pd
p = pd.read_csv('acq_params.tsv', sep='\t')
c = pd.read_csv('cohort.tsv', sep='\t')
m = c.merge(p, left_on='participant_id', right_on='sub')
m['protocol'] = m['serial'].astype(str) + '_tr' + m['tr'].astype(str) + '_sl' + m['nslices'].astype(str)
print(pd.crosstab(m['diagnosis'], m['protocol']), '\n')
print(pd.crosstab(m['diagnosis'], m['serial']), '\n')
from scipy.stats import chi2_contingency
print("protocol vs diagnosis chi2 p = %.4f" % chi2_contingency(pd.crosstab(m['diagnosis'], m['protocol']))[1])
m.to_csv('cohort_with_acq.tsv', sep='\t', index=False)

import numpy as np
b = np.loadtxt('raw/sub-50029/dwi/sub-50029_dwi.bval')
print(b.size, np.unique(np.round(b,-2)), 'n b0:', (b<50).sum())

protocol   35343_tr7.0_sl50  35343_tr8.4_sl60  35426_tr9.0_sl60
diagnosis                                                      
CONTROL                  12                29                 8
SCHZ                      1                24                24 

serial     35343  35426
diagnosis              
CONTROL       41      8
SCHZ          25     24 

protocol vs diagnosis chi2 p = 0.0001
32 [   0. 1000.] n b0: 1


In [7]:
import pandas as pd
df = pd.read_csv('participants.tsv', sep='\t')
elig = df[df['diagnosis'].isin(['CONTROL','SCHZ']) & df['dwi'].notna()]
print(pd.crosstab(elig['diagnosis'], elig['ScannerSerialNumber']))

ScannerSerialNumber  35343.0  35426.0
diagnosis                            
CONTROL                  100       23
SCHZ                      25       24


In [8]:
import json, glob, pandas as pd, os
df = pd.read_csv('participants.tsv', sep='\t')

def proto(path):
    d = json.load(open(path))
    return f"{d['DeviceSerialNumber']}_tr{d['RepetitionTime']}_sl{len(d.get('SliceTiming',[]))}"

rows = []
for f in glob.glob('json_probe/*_dwi.json'):
    rows.append({'participant_id': os.path.basename(f).replace('_dwi.json',''),
                 'protocol': proto(f)})
for f in glob.glob('raw/sub-*/dwi/*_dwi.json'):
    rows.append({'participant_id': os.path.basename(f).replace('_dwi.json',''),
                 'protocol': proto(f)})

pr = pd.DataFrame(rows).drop_duplicates('participant_id')
m = df.merge(pr, on='participant_id')
m = m[m['diagnosis'].isin(['CONTROL','SCHZ']) & m['dwi'].notna()]
m = m[m['participant_id'] != 'sub-50029']

print(pd.crosstab([m['diagnosis'], m['gender']], m['protocol']))
m.to_csv('eligible_with_protocol.tsv', sep='\t', index=False)

protocol          35343_tr7_sl50  35343_tr8.4_sl60  35426_tr9_sl60
diagnosis gender                                                  
CONTROL   F                    7                39              12
          M                   13                41              11
SCHZ      F                    0                 7               4
          M                    1                16              20


In [9]:
import pandas as pd, numpy as np
from scipy import stats

m = pd.read_csv('eligible_with_protocol.tsv', sep='\t')
scz  = m[m['diagnosis']=='SCHZ'].copy()
ctrl = m[m['diagnosis']=='CONTROL'].copy()
scz = scz.reindex(scz['age'].sub(scz['age'].mean()).abs().sort_values(ascending=False).index)

used, pairs, dropped = set(), [], []
for _, p in scz.iterrows():
    cand = ctrl[(ctrl['gender']==p['gender']) & (ctrl['protocol']==p['protocol'])
                & (~ctrl['participant_id'].isin(used))]
    if cand.empty:
        dropped.append(p['participant_id']); continue
    c = cand.iloc[(cand['age']-p['age']).abs().argmin()]
    used.add(c['participant_id'])
    pairs.append({'scz_id':p['participant_id'],'ctrl_id':c['participant_id'],
                  'protocol':p['protocol'],'gender':p['gender'],
                  'scz_age':p['age'],'ctrl_age':c['age'],
                  'diff':abs(p['age']-c['age'])})

pairs = pd.DataFrame(pairs)
print(f"pairs: {len(pairs)}   dropped: {len(dropped)}")
print("age diff  mean %.2f  max %d\n" % (pairs['diff'].mean(), pairs['diff'].max()))

sel = m[m['participant_id'].isin(pairs['scz_id']) | m['participant_id'].isin(pairs['ctrl_id'])]
print(sel.groupby('diagnosis')['age'].agg(['count','mean','std']), '\n')
print(pd.crosstab(sel['diagnosis'], sel['protocol']), '\n')
print(pd.crosstab(sel['diagnosis'], sel['gender']), '\n')

a = sel[sel['diagnosis']=='SCHZ']['age']; b = sel[sel['diagnosis']=='CONTROL']['age']
print("age  p = %.3f" % stats.ttest_ind(a,b).pvalue)
print("prot p = %.3f" % stats.chi2_contingency(pd.crosstab(sel['diagnosis'],sel['protocol']))[1])
print("ghost:\n", pd.crosstab(sel['diagnosis'], sel['ghost_NoGhost']))

sel.to_csv('cohort_final.tsv', sep='\t', index=False)
pairs.to_csv('matched_pairs_final.tsv', sep='\t', index=False)
print("\ndropped:", dropped)

pairs: 39   dropped: 9
age diff  mean 1.67  max 11

           count       mean       std
diagnosis                            
CONTROL       39  35.025641  8.609931
SCHZ          39  35.923077  9.476799 

protocol   35343_tr7_sl50  35343_tr8.4_sl60  35426_tr9_sl60
diagnosis                                                  
CONTROL                 1                23              15
SCHZ                    1                23              15 

gender      F   M
diagnosis        
CONTROL    11  28
SCHZ       11  28 

age  p = 0.663
prot p = 1.000
ghost:
 ghost_NoGhost  No_ghost  ghost
diagnosis                     
CONTROL              32      7
SCHZ                 22     17

dropped: ['sub-50077', 'sub-50076', 'sub-50060', 'sub-50083', 'sub-50081', 'sub-50059', 'sub-50066', 'sub-50075', 'sub-50051']
